In [11]:
# Step 1: Import All Required Libraries
import pandas as pd
import numpy as np
import pickle
import joblib
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

print("✅ All libraries imported!")

✅ All libraries imported!


In [12]:
# Step 2: Load the Vectorizer
print("🔍 ANALYZING GROUP MEMBER'S VECTORIZER...")

with open('tfidf_vectorizer.pkl', 'rb') as f:
    group_vectorizer = pickle.load(f)

print(f"✅ Vectorizer loaded: {type(group_vectorizer)}")

# Check vectorizer parameters
print(f"\n📊 VECTORIZER CONFIGURATION:")
print(f"Max features: {getattr(group_vectorizer, 'max_features', 'Not set')}")
print(f"Lowercase: {getattr(group_vectorizer, 'lowercase', 'Not set')}")
print(f"Stop words: {getattr(group_vectorizer, 'stop_words_', 'Not set')}")

🔍 ANALYZING GROUP MEMBER'S VECTORIZER...
✅ Vectorizer loaded: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>

📊 VECTORIZER CONFIGURATION:
Max features: 5000
Lowercase: True
Stop words: Not set


In [13]:
# Step 3: Vocabulary Quality Check
print(f"\n🎯 VOCABULARY QUALITY CHECK:")
feature_names = group_vectorizer.get_feature_names_out()
print(f"Total features: {len(feature_names)}")

# Analyze first and last features
print(f"First 20 features: {feature_names[:20].tolist()}")
print(f"Last 20 features: {feature_names[-20:].tolist()}")


🎯 VOCABULARY QUALITY CHECK:
Total features: 5000
First 20 features: ['adsbygoogle', 'beats', 'bengal', 'bengal beats', 'com', 'eআরক', 'of', 'sharetweet', 'the', 'অক', 'অক বর', 'অক ষর', 'অগ', 'অগ রগত', 'অঙ', 'অচ', 'অজ', 'অঞ', 'অঞ চল', 'অট']
Last 20 features: ['৭০', '৭০০', '৭১', '৭২', '৭৩', '৭৩তম', '৭৩তম অধ', '৭৫', '৭৯', '৮ট', '৮০', '৮২', '৮৩', '৮৫', '৮৬', '৮৮', '৯ট', '৯০', '৯৫', '৯৭']


In [14]:
# Step 4: Language Distribution Analysis
print(f"\n🌐 LANGUAGE ANALYSIS:")

bangla_count = sum(1 for word in feature_names if re.search(r'[ঀ-৿]', word))
english_count = sum(1 for word in feature_names if re.search(r'[a-zA-Z]', word))
number_count = sum(1 for word in feature_names if re.search(r'^\d+$', word))
mixed_count = len(feature_names) - bangla_count - english_count - number_count

print(f"Pure Bangla words: {bangla_count} ({bangla_count/len(feature_names)*100:.1f}%)")
print(f"Pure English words: {english_count} ({english_count/len(feature_names)*100:.1f}%)")
print(f"Numbers only: {number_count} ({number_count/len(feature_names)*100:.1f}%)")
print(f"Mixed/Other: {mixed_count} ({mixed_count/len(feature_names)*100:.1f}%)")


🌐 LANGUAGE ANALYSIS:
Pure Bangla words: 4992 (99.8%)
Pure English words: 9 (0.2%)
Numbers only: 109 (2.2%)
Mixed/Other: -110 (-2.2%)


In [15]:
# Step 5: Test on Sample Bangla Text
print("\n🧪 TESTING ON SAMPLE BANGLA TEXT...")

# Sample clean Bangla news text
sample_bangla_text = [
    "রাজধানী ঢাকায় আজ একটি ভয়াবহ সড়ক দুর্ঘটনা ঘটেছে",
    "প্রধানমন্ত্রী শেখ হাসিনা সংসদে গুরুত্বপূর্ণ বক্তব্য দিয়েছেন",
    "বাংলাদেশের অর্থনীতি স্থিতিশীল হচ্ছে বলে সরকার দাবি করেছে"
]

# Transform using the vectorizer
sample_vectorized = group_vectorizer.transform(sample_bangla_text)
print(f"Sample output shape: {sample_vectorized.shape}")

# Check if it captures meaningful features
print(f"\n📝 FEATURE ACTIVATION CHECK:")
for i, text in enumerate(sample_bangla_text):
    # Get non-zero features for this text
    non_zero_indices = sample_vectorized[i].nonzero()[1]
    if len(non_zero_indices) > 0:
        activated_features = [feature_names[idx] for idx in non_zero_indices[:5]]  # First 5
        print(f"Text {i+1}: {text[:50]}...")
        print(f"  Activated features: {activated_features}")
    else:
        print(f"Text {i+1}: NO FEATURES ACTIVATED! 🚨")


🧪 TESTING ON SAMPLE BANGLA TEXT...
Sample output shape: (3, 5000)

📝 FEATURE ACTIVATION CHECK:
Text 1: রাজধানী ঢাকায় আজ একটি ভয়াবহ সড়ক দুর্ঘটনা ঘটেছে...
  Activated features: ['আজ', 'একট', 'ঘট', 'ঘটন', 'ঘটন ঘট']
Text 2: প্রধানমন্ত্রী শেখ হাসিনা সংসদে গুরুত্বপূর্ণ বক্তব্...
  Activated features: ['তব', 'নমন', 'বক', 'বক তব', 'বপ']
Text 3: বাংলাদেশের অর্থনীতি স্থিতিশীল হচ্ছে বলে সরকার দাবি...
  Activated features: ['অর', 'অর থন', 'কর', 'থন', 'বল']


In [16]:
# Step 6: Test on Actual Data
print("\n📊 TESTING ON YOUR ACTUAL DATA...")

# Load your data first
df = pd.read_csv('../data/processed_data.csv')

# Transform your actual data
X_actual = group_vectorizer.transform(df['text'])
y_actual = df['label']

print(f"Actual data shape: {X_actual.shape}")
print(f"Data sparsity: {(1 - (X_actual.nnz / (X_actual.shape[0] * X_actual.shape[1]))) * 100:.1f}% empty")

# Check feature distribution
print(f"\n📈 FEATURE USAGE STATS:")
non_zero_per_article = X_actual.getnnz(axis=1)
print(f"Avg features per article: {non_zero_per_article.mean():.1f}")
print(f"Min features per article: {non_zero_per_article.min()}")
print(f"Max features per article: {non_zero_per_article.max()}")
print(f"Articles with < 5 features: {(non_zero_per_article < 5).sum()}")
print(f"Articles with 0 features: {(non_zero_per_article == 0).sum()}")


📊 TESTING ON YOUR ACTUAL DATA...
Actual data shape: (8501, 5000)
Data sparsity: 97.3% empty

📈 FEATURE USAGE STATS:
Avg features per article: 134.4
Min features per article: 2
Max features per article: 962
Articles with < 5 features: 3
Articles with 0 features: 0


In [17]:
# Step 7: Final Quality Assessment
print("\n🎯 FINAL QUALITY ASSESSMENT:")

# Scoring system
quality_score = 0
max_score = 10

# 1. Bangla content check
bangla_ratio = bangla_count / len(feature_names)
if bangla_ratio > 0.8:
    print("✅ Excellent: >80% Bangla features")
    quality_score += 3
elif bangla_ratio > 0.6:
    print("⚠️  Acceptable: 60-80% Bangla features")
    quality_score += 2
else:
    print("❌ Poor: <60% Bangla features")
    quality_score += 0

# 2. Feature activation check
avg_features = non_zero_per_article.mean()
if avg_features > 50:
    print("✅ Excellent: Good feature density")
    quality_score += 3
elif avg_features > 20:
    print("⚠️  Acceptable: Moderate feature density")
    quality_score += 2
else:
    print("❌ Poor: Low feature density")
    quality_score += 0

# 3. Zero-feature articles check
zero_feature_articles = (non_zero_per_article == 0).sum()
if zero_feature_articles == 0:
    print("✅ Excellent: No empty articles")
    quality_score += 2
elif zero_feature_articles < 10:
    print("⚠️  Acceptable: Few empty articles")
    quality_score += 1
else:
    print("❌ Poor: Many empty articles")
    quality_score += 0

# 4. English contamination check
english_ratio = english_count / len(feature_names)
if english_ratio < 0.05:
    print("✅ Excellent: Minimal English contamination")
    quality_score += 2
elif english_ratio < 0.1:
    print("⚠️  Acceptable: Some English contamination")
    quality_score += 1
else:
    print("❌ Poor: High English contamination")
    quality_score += 0

print(f"\n🏆 OVERALL QUALITY SCORE: {quality_score}/{max_score}")

if quality_score >= 8:
    print("🎉 EXCELLENT - Use this vectorizer!")
elif quality_score >= 5:
    print("⚠️  ACCEPTABLE - Could be better but usable")
else:
    print("🚨 POOR - Consider creating a new one")


🎯 FINAL QUALITY ASSESSMENT:
✅ Excellent: >80% Bangla features
✅ Excellent: Good feature density
✅ Excellent: No empty articles
✅ Excellent: Minimal English contamination

🏆 OVERALL QUALITY SCORE: 10/10
🎉 EXCELLENT - Use this vectorizer!


In [18]:
# Step 8: Quick Model Test to Verify Everything Works
print("\n🔬 QUICK MODEL VERIFICATION TEST:")

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_actual, y_actual, test_size=0.2, random_state=42, stratify=y_actual
)

# Quick model test
test_model = MultinomialNB()
test_model.fit(X_train, y_train)
test_accuracy = test_model.score(X_test, y_test)

print(f"Quick Naive Bayes test accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

if test_accuracy > 0.85:
    print("✅ Vectorizer is producing good results!")
else:
    print("⚠️  Vectorizer might have issues")


🔬 QUICK MODEL VERIFICATION TEST:
Quick Naive Bayes test accuracy: 0.9359 (93.59%)
✅ Vectorizer is producing good results!
